<a href="https://colab.research.google.com/github/haydenkellington/EclipseBaseballSpring26/blob/main/Spring26BaseballProj_FINAL_(7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Title

Website-pitch-set modeling copy: FF, SI, FC, SL, ST, CU, CH, FS are the only allowed actions from MDP preparation onward.

**Team Members:** Abhishai Ganta, Anay Takkallapalli, Dhruv Sehgal

**Date:** Spring Quarter 2026

---

## Research Question
Can a reinforcement learning agent learn pitch sequencing policies that reduce expected run value compared to historical MLB pitch selection?

**Sub-questions:**
- RL vs MLB performance (run value)
- Effect of sequencing (prev pitch vs none)
- Key pitch sequences and their outcomes

**Expected Outcomes:**
- We expect reinforcement learning pitch sequencing to slightly outperform MLB strategies by reducing run value and using less predictable pitch patterns.

---

## Data Source

**Dataset Name:** pybaseball

**Link:** https://github.com/jldbc/pybaseball

**Description:**
- TODO: Briefly describe the dataset
- Number of observations: [TODO]
- Number of features: [TODO]
- Key variables: [TODO: List important columns]
- Time period covered: [TODO]
- Data collection method: [TODO]

**Citation:**
LeDoux, J. (2017, July 27). Introducing pybaseball: an Open Source Package for Baseball Data Analysis. https://jamesrledoux.com/projects/open-source/introducing-pybaseball/

---

## Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install pybaseball
import sklearn
import scipy

# TODO: Add additional imports as needed
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
import pybaseball as pyb
from pybaseball import statcast
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss, accuracy_score


# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# For reproducibility
np.random.seed(68)

print("Imports successful!")

---

## Data Loading

**TODO:** Load your dataset and perform initial inspection

In [ ]:
import os
import pandas as pd
import pybaseball as pyb

# Enable pybaseball caching (avoids redundant downloads)
pyb.cache.enable()

# Try Google Drive parquet cache first — loads in seconds instead of ~10 min
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = '/content/drive/MyDrive/statcast_2024_2025.parquet'
    if os.path.exists(data_path):
        print("Loading from Google Drive parquet cache...")
        df = pd.read_parquet(data_path)
        print(f"Loaded instantly. Shape: {df.shape}")
    else:
        raise FileNotFoundError("No parquet cache yet")
except Exception:
    # Fresh pull from Statcast API (~10 min one-time cost)
    print("Fetching 2024 season from Statcast API (one-time, ~10 min)...")
    df = pyb.statcast(start_dt='2024-03-03', end_dt='2024-11-30')
    # Save for next time
    try:
        df.to_parquet(data_path, index=False)
        print("Saved to Google Drive for next run.")
    except Exception:
        pass

if df is not None:
    print(f"Dataset shape: {df.shape}")
    display(df.head(3))


In [ ]:
# TODO: Examine dataset structure
if df is not None:
    print("Dataset Info:")
    df.info()

    print("\n" + "="*50)
    print("Summary Statistics:")
    display(df.describe())

    print("\n" + "="*50)
    print("Data Types:")
    display(df.dtypes)

**Initial Observations:**

TODO: Document your first impressions of the data:
- Are there any obvious issues?
- Do the data types look correct?
- Are there missing values?
- Do the value ranges make sense?

---

## Data Cleaning

**TODO:** Clean and preprocess the data

### Missing Values Analysis

In [ ]:
# TODO: Check for missing values
if df is not None:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100

    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    }).sort_values('Percentage', ascending=False)

    print("Missing Values Summary:")
    display(missing_df[missing_df['Missing Count'] > 0])

    # Visualize missing data
    if missing.sum() > 0:
        plt.figure(figsize=(10, 6))
        missing_df[missing_df['Missing Count'] > 0]['Percentage'].plot(kind='barh')
        plt.xlabel('Percentage Missing')
        plt.title('Missing Values by Column')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Data Cleaning ─────────────────────────────────────────────────────────────
# CRITICAL: do NOT fill on_1b/on_2b/on_3b with 0 here.
# Statcast stores runner presence as the runner's player ID (a float) or NaN
# when the base is empty. Cell 21 converts correctly with .notna().
# Filling with 0 first makes 0 (not NaN) look like a runner is present.
#
# Steps:
#   1. Drop deprecated/empty legacy columns
#   2. Leave on_1b/2b/3b as-is — NaN = empty base
#   3. Drop rows missing pitch_type or delta_run_exp (critical for reward)
#   4. Fill plate_x/z with median (tracking errors, small %)
#   5. Clip release_speed to [40, 105] mph

if df is not None:
    deprecated = [
        'break_angle_deprecated', 'break_length_deprecated',
        'spin_dir', 'spin_rate_deprecated',
        'umpire', 'sv_id', 'tfs_deprecated', 'tfs_zulu_deprecated'
    ]
    df = df.drop(columns=[c for c in deprecated if c in df.columns])
    print(f"After dropping deprecated cols: {df.shape}")

    # DO NOT fill runners here — leave NaN intact for correct notna() in Cell 21

    critical = ['pitch_type', 'delta_run_exp', 'balls', 'strikes']
    before = len(df)
    df = df.dropna(subset=[c for c in critical if c in df.columns])
    print(f"Dropped {before - len(df):,} rows missing critical fields → {len(df):,} remain")

    for col in ['plate_x', 'plate_z']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].median())

    if 'release_speed' in df.columns:
        df['release_speed'] = pd.to_numeric(df['release_speed'], errors='coerce')
        before = len(df)
        df = df[df['release_speed'].isna() | df['release_speed'].between(40, 105)]
        print(f"Removed {before - len(df):,} rows with implausible release_speed")

    print(f"\nCleaned df shape: {df.shape}")
    for col in ['on_1b', 'on_2b', 'on_3b']:
        if col in df.columns:
            print(f"  {col}: {df[col].isna().sum():,} NaN (= empty bases) — correct ✓")


### Duplicate Detection

In [ ]:
# TODO: Check for duplicates
if df is not None:
    duplicates = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicates}")

    if duplicates > 0:
        print("\nDuplicate rows:")
        display(df[df.duplicated(keep=False)])

        # TODO: Decide whether to keep or remove duplicates
        # df_clean = df_clean.drop_duplicates()

### Data Type Conversions

In [ ]:
# TODO: Convert data types as needed
# Examples:
# df_clean['date_column'] = pd.to_datetime(df_clean['date_column'])
# df_clean['category_column'] = df_clean['category_column'].astype('category')
# df_clean['numeric_column'] = pd.to_numeric(df_clean['numeric_column'], errors='coerce')

pass

### Outlier Detection

In [ ]:
# TODO: Detect outliers in numeric columns
# Common methods:
# 1. IQR method
# 2. Z-score method
# 3. Visual inspection with box plots

# Example: Box plots for numeric columns
if df is not None:
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        # TODO: Create box plots for numeric columns
        # fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(10, 3*len(numeric_cols)))
        # for i, col in enumerate(numeric_cols):
        #     df.boxplot(column=col, ax=axes[i])
        # plt.tight_layout()
        # plt.show()
        pass

### Feature Engineering

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
# Builds all features needed for the MDP. Both agents read from `df` after this.
#
# STATE FEATURES (used by both agents):
#   balls, strikes          — the count
#   outs_when_up → outs     — how many outs (0/1/2) when batter stepped up
#   on_1b, on_2b, on_3b    — 3 separate binary runner flags (not compressed)
#   batter_righty           — 1=RHB, 0=LHB
#   pitcher_righty          — 1=RHP, 0=LHP
#
# RUNNER ENCODING:
#   Statcast: NaN = base empty, player_id float = runner present.
#   We use .notna() to convert to 0/1. Do NOT fill NaN first (Cell 13 correct).
#   We keep all 3 as separate binary dimensions preserving all 8 base combos.
#   Bases loaded ≠ runner on 2nd only — different pitching strategy entirely.
#
# HISTORY FEATURE (augmented agent only):
#   prev_pitch_type → prev_pitch_type (string, used to build prev_action array)
#   This is the ONLY history dimension the augmented agent uses.
#   No location, no TTO — those add state-space cost without helping the
#   core research question: does knowing the previous pitch type improve policy?
#
# REWARD:
#   delta_run_exp clipped to [-0.5, 0.5]. Raw values kept as delta_run_exp_raw.
#   Clipping prevents grand slam outliers from dominating Q-value updates.

if df is not None:
    base_cols = [
        'game_date', 'game_pk', 'at_bat_number', 'pitch_number',
        'pitch_type', 'description', 'events',
        'balls', 'strikes', 'outs_when_up',
        'stand', 'p_throws',
        'plate_x', 'plate_z', 'release_speed',
        'release_pos_x', 'release_pos_z',
        'on_1b', 'on_2b', 'on_3b',
        'delta_run_exp',
    ]
    selected_cols = [c for c in base_cols if c in df.columns]
    df_clean = df[selected_cols].copy()

    # At-bat ID + chronological sort
    df_clean['at_bat_id'] = (
        df_clean['game_pk'].astype(str).fillna('') + '_' +
        df_clean['at_bat_number'].astype(str).fillna('')
    )
    for col in ['game_pk', 'at_bat_number', 'pitch_number']:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    df_clean = df_clean.sort_values(
        ['game_pk', 'at_bat_number', 'pitch_number'], kind='stable'
    ).reset_index(drop=True)

    # Count state
    df_clean['balls']   = df_clean['balls'].astype(int)
    df_clean['strikes'] = df_clean['strikes'].astype(int)
    df_clean['is_first_pitch'] = (
        (df_clean['balls'] == 0) & (df_clean['strikes'] == 0)
    ).astype(int)
    df_clean['count_state']   = df_clean['balls'].astype(str) + '-' + df_clean['strikes'].astype(str)
    df_clean['count_numeric'] = df_clean['balls'] * 10 + df_clean['strikes']

    # Outs (0/1/2)
    if 'outs_when_up' in df_clean.columns:
        df_clean['outs'] = (
            pd.to_numeric(df_clean['outs_when_up'], errors='coerce')
            .fillna(0).astype(int).clip(0, 2)
        )
    else:
        df_clean['outs'] = 0

    # Runner presence — CORRECT: NaN = empty base
    # .notna() returns 1 if a player ID is present, 0 if NaN (empty)
    for col in ['on_1b', 'on_2b', 'on_3b']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].notna().astype(int)

    # Keep EDA aliases
    df_clean['is_on_1b'] = df_clean.get('on_1b', 0)
    df_clean['is_on_2b'] = df_clean.get('on_2b', 0)
    df_clean['is_on_3b'] = df_clean.get('on_3b', 0)
    df_clean['base_state'] = (
        df_clean['is_on_1b'] * 1 +
        df_clean['is_on_2b'] * 2 +
        df_clean['is_on_3b'] * 4
    )

    # Handedness — string for display, binary int for MDP state
    if 'stand' in df_clean.columns:
        df_clean['batter_side']   = df_clean['stand'].fillna('U')
        df_clean['batter_righty'] = (df_clean['stand'] == 'R').astype(int)
    else:
        df_clean['batter_side']   = 'U'
        df_clean['batter_righty'] = 1
    if 'p_throws' in df_clean.columns:
        df_clean['pitcher_hand']   = df_clean['p_throws'].fillna('U')
        df_clean['pitcher_righty'] = (df_clean['p_throws'] == 'R').astype(int)
    else:
        df_clean['pitcher_hand']   = 'U'
        df_clean['pitcher_righty'] = 1

    # Clip extreme delta_run_exp — prevents grand slam outliers from dominating Q-values
    if 'delta_run_exp' in df_clean.columns:
        df_clean['delta_run_exp_raw'] = df_clean['delta_run_exp'].copy()
        df_clean['delta_run_exp']     = df_clean['delta_run_exp'].clip(-0.5, 0.5)

    # Previous pitch features (within-AB shift — first pitch of AB gets NaN)
    for col in ['pitch_type', 'description', 'events', 'delta_run_exp']:
        if col in df_clean.columns:
            df_clean[f'prev_{col}'] = df_clean.groupby('at_bat_id')[col].shift(1)

    # Strike zone flag (for EDA)
    if {'plate_x', 'plate_z'}.issubset(df_clean.columns):
        df_clean['plate_x'] = pd.to_numeric(df_clean['plate_x'], errors='coerce')
        df_clean['plate_z'] = pd.to_numeric(df_clean['plate_z'], errors='coerce')
        df_clean['in_strike_zone'] = (
            df_clean['plate_x'].between(-0.83, 0.83) &
            df_clean['plate_z'].between(1.5, 3.5)
        ).fillna(False).astype(int)

    display_columns = [
        'game_pk', 'at_bat_number', 'at_bat_id', 'pitch_number', 'is_first_pitch',
        'balls', 'strikes', 'outs',
        'count_state', 'count_numeric',
        'on_1b', 'on_2b', 'on_3b',
        'is_on_1b', 'is_on_2b', 'is_on_3b', 'base_state',
        'batter_side', 'pitcher_hand', 'batter_righty', 'pitcher_righty',
        'pitch_type', 'description', 'events',
        'delta_run_exp', 'delta_run_exp_raw',
        'prev_pitch_type', 'prev_description', 'prev_events', 'prev_delta_run_exp',
        'plate_x', 'plate_z', 'in_strike_zone',
        'release_speed', 'release_pos_x', 'release_pos_z',
    ]
    display_columns = [c for c in display_columns if c in df_clean.columns]
    df = df_clean[display_columns].copy()

    print(f"Feature-engineered df: {df.shape}")
    print(f"Runner occupancy: on_1b={df['on_1b'].mean():.1%}  on_2b={df['on_2b'].mean():.1%}  on_3b={df['on_3b'].mean():.1%}")
    print(f"Outs distribution: {df['outs'].value_counts().sort_index().to_dict()}")
    print(f"prev_pitch_type present: {'prev_pitch_type' in df.columns} ✓")
    display(df.head(3))
else:
    print("df is None — skipping feature engineering")


In [ ]:
# TODO: Save cleaned dataset (optional)
# df_clean.to_csv('data/cleaned_data.csv', index=False)
# print("Cleaned data saved!")

**Cleaning Summary:**

TODO: Document what cleaning steps were performed and why:
- Missing values: [strategy used]
- Duplicates: [action taken]
- Outliers: [how handled]
- Feature engineering: [new features created]

---

## Exploratory Data Analysis

**TODO:** Explore the data to understand patterns, relationships, and distributions

Graphs

### Bivariate Analysis

### Bivariate Analysis

In [ ]:
pitch_by_count = df.groupby(['count_state', 'pitch_type']).size().reset_index(name='count')
#sns.barplot(data=pitch_by_count, x='count_state', y='count', hue='pitch_type')

In [ ]:
# top 8 most common pitch types
top_pitches = df['pitch_type'].value_counts().head(8).index

filtered = df[
    df['pitch_type'].isin(top_pitches) &
    df['prev_pitch_type'].isin(top_pitches)
]

transition = pd.crosstab(filtered['prev_pitch_type'], filtered['pitch_type'])

#plt.figure(figsize=(8, 6))
#sns.heatmap(transition,annot=True,fmt='d',cmap='Blues',linewidths=0.5)

#plt.title('Pitch Transition Frequencies (Top Pitch Types)')
#plt.xlabel('Current Pitch Type')
#plt.ylabel('Previous Pitch Type')
#plt.tight_layout()
#plt.show()

In [ ]:
# ── Pitch Sequencing Transition Heatmaps by Count Group ──────────────────
def count_group(count):
    if count in ['2-0', '3-0', '3-1', '2-1']:
        return 'Hitter Count'
    elif count in ['0-2', '1-2', '0-1']:
        return 'Pitcher Count'
    else:
        return 'Neutral Count'

df['count_group'] = df['count_state'].apply(count_group)

top_pitches = df['pitch_type'].value_counts().head(8).index

filtered = df[
    df['pitch_type'].isin(top_pitches) &
    df['prev_pitch_type'].isin(top_pitches) &
    df['prev_pitch_type'].notna()
]

groups = ['Hitter Count', 'Neutral Count', 'Pitcher Count']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, group in zip(axes, groups):
    subset = filtered[filtered['count_group'] == group]

    if subset.empty:
        ax.set_title(f'{group}\n(No Data)')
        ax.axis('off')
        continue

    transition = pd.crosstab(
        subset['prev_pitch_type'],
        subset['pitch_type'],
        normalize='index'
    )

    sns.heatmap(transition, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.5, ax=ax)
    ax.set_title(group)
    ax.set_xlabel('Current Pitch')
    ax.set_ylabel('Previous Pitch')

plt.suptitle('Pitch Sequencing Transition Probabilities by Count Situation', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
df = df.reset_index(drop=True)
#sns.boxplot(data=df, x='pitch_type', y='delta_run_exp')

In [ ]:

# remove missing values
plot_df = df[
    df['prev_pitch_type'].notna() &
    df['pitch_type'].notna() &
    df['delta_run_exp'].notna()
].copy()

# create sequence label
plot_df['sequence'] = plot_df['prev_pitch_type'] + ' → ' + plot_df['pitch_type']

# top 15 most common sequences
top_sequences = plot_df['sequence'].value_counts().head(15).index

plot_df = plot_df[plot_df['sequence'].isin(top_sequences)]

# average run expectancy change
avg_seq = (
    plot_df.groupby('sequence')['delta_run_exp']
    .mean()
    .sort_values()
    .reset_index()
)

#plt.figure(figsize=(12, 7))

#sns.barplot(
 #   data=avg_seq,
 #   x='delta_run_exp',
 #   y='sequence'
#)

#plt.title('Average Run Expectancy Change by Pitch Sequence')
#plt.xlabel('Average Delta Run Expectancy')
#plt.ylabel('Pitch Sequence (Previous → Current)')
#plt.tight_layout()
#plt.show()

In [ ]:
# ── Multivariate Relationships ────────────────────────────────────────────
# FIX (Bug 9): Previously had an indented 'pass' inside a commented-out if-block,
# causing IndentationError. Fixed by replacing with real correlation analysis.

if df is not None:
    # Correlation matrix among key numeric MDP features
    numeric_cols = ['balls', 'strikes', 'is_on_1b', 'is_on_2b', 'is_on_3b',
                    'batter_righty', 'pitcher_righty', 'delta_run_exp',
                    'release_speed', 'plate_x', 'plate_z']
    numeric_cols = [c for c in numeric_cols if c in df.columns]

    corr = df[numeric_cols].corr()

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                linewidths=0.5, center=0)
    plt.title('Correlation Matrix: MDP State Features')
    plt.tight_layout()
    plt.show()


**EDA Findings:**

TODO: Summarize key insights from your exploratory analysis:
- What are the main patterns in the data?
- Are there any unexpected findings?
- Which variables seem most relevant to your research question?
- Are there any data quality issues that need addressing?

---

## Modeling and Analysis

Our core modeling approach is a **tabular Q-learning** reinforcement learning agent that treats each at-bat pitch as a decision step in a Markov Decision Process (MDP). We also train a **history-augmented** variant that includes the previous pitch in the state vector, allowing the agent to learn tunneling effects. For comparison, we train an **XGBoost run-value predictor** as a supervised baseline, and we evaluate all models against the actual MLB decisions from the test set.

### MDP Formulation

| MDP Component | Definition |
|---|---|
| **State S** | (balls, strikes, on_1b, on_2b, on_3b, batter_righty, pitcher_righty, pitch_number_in_ab) |
| **History-Aug. State S'** | S + (prev_pitch_type) |
| **Action A** | Pitch type to throw (FF, SL, CH, CU, SI, FC, FS, ST, SV, KC) |
| **Reward R** | Negative `delta_run_exp` — pitcher wants to *minimize* run expectancy, so we negate it |
| **Transition** | Next state is the next row in the same at-bat, or terminal if at-bat ends |
| **Discount γ** | 0.95 — future pitch outcomes slightly discounted |

**Why Q-learning?** The state space is discrete and relatively small (a few thousand unique states), making tabular Q-learning tractable, interpretable, and fast to train on our dataset size.

### Data Preparation for Modeling

In [ ]:
# ── MDP Data Preparation ─────────────────────────────────────────────────────
# Selects and type-enforces exactly the columns both agents need.
#
# STATE DIMENSIONS (shared by both agents):
#   balls, strikes, outs        — game count state
#   on_1b, on_2b, on_3b        — runner presence (3 separate binary flags)
#   batter_righty, pitcher_righty — handedness matchup
#
# This gives: 4×3×3×2×2×2×2×2 = 1,152 theoretical states.
# With 1.2M training pitches → avg 1,042 obs/state. Excellent density.
# Every state is well-covered — no fallback needed in common situations.
#
# HISTORY FEATURE: prev_pitch_type — used to build aug_state_arr in Cell 39.
# It is NOT in STATE_COLS (that would make the memoryless agent history-aware).
# It is appended only in the augmented state array construction.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(68)

MDP_COLS = [
    'game_pk', 'at_bat_number', 'pitch_number', 'at_bat_id',
    # State dimensions
    'balls', 'strikes', 'outs',
    'on_1b', 'on_2b', 'on_3b',
    'batter_righty', 'pitcher_righty',
    # Previous pitch (for augmented agent only)
    'prev_pitch_type',
    # MDP label + reward
    'pitch_type', 'delta_run_exp',
    # EDA context
    'description', 'events',
]

available = [c for c in MDP_COLS if c in df.columns]
missing   = [c for c in MDP_COLS if c not in df.columns]
if missing:
    print(f"WARNING — MDP columns not in df: {missing}")
mdp_df = df[available].copy()

# Type enforcement
for col in ['on_1b', 'on_2b', 'on_3b']:
    if col in mdp_df.columns:
        mdp_df[col] = mdp_df[col].fillna(0).astype(int).clip(0, 1)
for col in ['batter_righty', 'pitcher_righty']:
    if col in mdp_df.columns:
        mdp_df[col] = mdp_df[col].fillna(1).astype(int)
if 'outs' in mdp_df.columns:
    mdp_df['outs'] = mdp_df['outs'].fillna(0).astype(int).clip(0, 2)

# Drop rows missing critical fields
before = len(mdp_df)
mdp_df = mdp_df.dropna(subset=['pitch_type', 'delta_run_exp', 'balls', 'strikes'])
print(f"Dropped {before - len(mdp_df):,} rows with missing critical fields")

# Restrict the modeling action space to the same pitch types used by the website.
# This prevents rare/noisy pitch codes from entering training or export while keeping FS.
# Website/dashboard pitch set used for modeling and deployment
# Keep this aligned with backend/model/pitch_mappings.py.
WEBSITE_PITCHES = ['FF', 'SI', 'FC', 'SL', 'ST', 'CU', 'CH', 'FS']
WEBSITE_PITCH_SET = set(WEBSITE_PITCHES)
before_pitch_filter = len(mdp_df)
mdp_df = mdp_df[mdp_df['pitch_type'].isin(WEBSITE_PITCH_SET)].copy()
print(
    f"Kept website pitch types {WEBSITE_PITCHES}: "
    f"{len(mdp_df):,} rows remain; removed {before_pitch_filter - len(mdp_df):,}"
)

# Sort chronologically
sort_cols = [c for c in ['game_pk', 'at_bat_number', 'pitch_number'] if c in mdp_df.columns]
for col in sort_cols:
    mdp_df[col] = pd.to_numeric(mdp_df[col], errors='coerce')
mdp_df = mdp_df.sort_values(sort_cols, kind='stable').reset_index(drop=True)

print(f"MDP dataset: {mdp_df.shape}")
print(f"Unique pitch types: {sorted(mdp_df['pitch_type'].unique())}")
print("Website pitch set applied from modeling start, including FS.")
for col in ['balls', 'strikes', 'outs', 'on_1b', 'on_2b', 'on_3b']:
    if col in mdp_df.columns:
        print(f"  {col}: unique values = {sorted(mdp_df[col].unique())}")

# Verify runner encoding is correct (not all 1s)
for col in ['on_1b', 'on_2b', 'on_3b']:
    rate = mdp_df[col].mean()
    status = "✓" if 0.05 < rate < 0.6 else "⚠ CHECK"
    print(f"  {col} occupancy: {rate:.1%} {status}")


In [ ]:
# ── Step 2: Encode pitch types as integers ────────────────────────────────
# Q-table actions are integer indices into the pitch type list

PITCH_TYPES = [p for p in WEBSITE_PITCHES if p in set(mdp_df['pitch_type'].unique())]
PITCH_TO_IDX = {p: i for i, p in enumerate(PITCH_TYPES)}
IDX_TO_PITCH = {i: p for p, i in PITCH_TO_IDX.items()}
N_ACTIONS = len(PITCH_TYPES)

mdp_df['action'] = mdp_df['pitch_type'].map(PITCH_TO_IDX)

print(f"Number of website pitch type actions: {N_ACTIONS}")
print("Pitch index mapping:")
for p, i in PITCH_TO_IDX.items():
    print(f"  {i}: {p}")


In [ ]:
# ── Build MDP Transition Arrays ───────────────────────────────────────────────
#
# STATE VECTOR — Memoryless Agent (8 dimensions):
#   balls(4) × strikes(3) × outs(3)
#   × on_1b(2) × on_2b(2) × on_3b(2)
#   × batter_righty(2) × pitcher_righty(2)
#   = 4×3×3×2×2×2×2×2 = 1,152 theoretical max states
#
# AUGMENTED STATE — History Agent (9 dimensions = 8 base + prev_pitch):
#   Same base state with prev_pitch_action_idx appended as last column.
#   prev_pitch = -1 sentinel for first pitch of AB (no prior pitch).
#   With 18 pitch types: adds 19 values (0-17 + sentinel -1) → state space
#   becomes 1,152 × 19 = 21,888 — still very well covered by the data.
#
# WHY THIS DESIGN CLEANLY TESTS THE HYPOTHESIS:
#   Memoryless: learns from (count, runners, handedness) alone
#   Augmented:  learns from exactly the same base + what was thrown last
#   The difference in Q-advantage between the two directly measures
#   how much pitch sequencing history is worth.
#
# REWARD: -delta_run_exp (already clipped to [-0.5, 0.5] in Cell 21)
#   Positive reward = pitcher reduced run expectancy = good
#   Negative reward = pitcher increased run expectancy = bad

STATE_COLS = [c for c in [
    'balls', 'strikes', 'outs',
    'on_1b', 'on_2b', 'on_3b',
    'batter_righty', 'pitcher_righty',
] if c in mdp_df.columns]

# State space calculation
dim_sizes = {
    'balls':4, 'strikes':3, 'outs':3,
    'on_1b':2, 'on_2b':2, 'on_3b':2,
    'batter_righty':2, 'pitcher_righty':2,
}
state_space = 1
for c in STATE_COLS:
    state_space *= dim_sizes.get(c, 2)

print(f"Memoryless STATE_COLS ({len(STATE_COLS)}): {STATE_COLS}")
print(f"Theoretical max states: {state_space:,}")
print(f"Augmented max states:   {state_space * (N_ACTIONS + 1):,}  (+1 for sentinel)")
# Note: data density printed in Cell 40 after the train/test split

# At-bat group key
if 'game_pk' in mdp_df.columns and 'at_bat_number' in mdp_df.columns:
    mdp_df['_ab_key'] = mdp_df['game_pk'].astype(str) + '_' + mdp_df['at_bat_number'].astype(str)
else:
    mdp_df['_ab_key'] = mdp_df['at_bat_id']

grp = mdp_df.groupby('_ab_key', sort=False)

# Done flag: last pitch of each at-bat
ab_sizes = grp['_ab_key'].transform('size')
ab_rank  = grp.cumcount()
mdp_df['_done'] = (ab_rank == ab_sizes - 1).astype(int)

# Reward: negate delta_run_exp (pitcher wants to reduce run expectancy)
# Clipping already applied in Cell 21; second clip catches any edge cases
mdp_df['_reward'] = (-mdp_df['delta_run_exp']).clip(-0.5, 0.5).astype(np.float32)

# Next-state via vectorized shift within AB
# Terminal rows: next_state = current (placeholder — never used when done=True)
for col in STATE_COLS + ['action']:
    shifted = grp[col].shift(-1)
    mdp_df[f'_next_{col}'] = shifted.where(mdp_df['_done'] == 0, mdp_df[col])

# Previous action: -1 sentinel for first pitch of AB
mdp_df['_prev_action'] = grp['action'].shift(1).fillna(-1).astype(int)

# Convert to numpy arrays
state_arr      = mdp_df[STATE_COLS].values.astype(np.int32)
next_state_arr = mdp_df[[f'_next_{c}' for c in STATE_COLS]].values.astype(np.int32)
prev_act_arr   = mdp_df['_prev_action'].values.astype(np.int32)
action_arr     = mdp_df['action'].values.astype(np.int32)
reward_arr     = mdp_df['_reward'].values.astype(np.float32)
done_arr       = mdp_df['_done'].values.astype(bool)

# Augmented state: base + prev_pitch appended as last column
aug_state_arr      = np.hstack([state_arr,      prev_act_arr.reshape(-1, 1)])
aug_next_state_arr = np.hstack([next_state_arr, action_arr.reshape(-1, 1)])

print(f"\nstate_arr:     {state_arr.shape}   ({state_arr.shape[1]}D memoryless)")
print(f"aug_state_arr: {aug_state_arr.shape}  ({aug_state_arr.shape[1]}D augmented)")
print(f"Total pitches: {len(action_arr):,}")
print(f"Terminal rows: {done_arr.sum():,}  (done fraction: {done_arr.mean():.3f})")
print(f"Reward range:  [{reward_arr.min():.3f}, {reward_arr.max():.3f}]")

# Sanity: reward by pitch type (all should be small, near zero)
print(f"\nMean reward by pitch type:")
for pt in sorted(PITCH_TYPES):
    mask = mdp_df['pitch_type'] == pt
    if mask.sum() > 1000:
        print(f"  {pt}: {mdp_df.loc[mask,'_reward'].mean():+.4f}  (n={mask.sum():,})")


In [ ]:
# ── Train/Test Split (80/20 by at-bat) ───────────────────────────────────────
# Split at the AT-BAT level to prevent data leakage — pitches from the same
# at-bat must all be in the same split.

from sklearn.model_selection import train_test_split

unique_abs = mdp_df['_ab_key'].unique()
train_abs, test_abs = train_test_split(unique_abs, test_size=0.2, random_state=68)
train_ab_set = set(train_abs)

train_mask = mdp_df['_ab_key'].isin(train_ab_set).values
test_mask  = ~train_mask

print(f"Unique at-bats:  {len(unique_abs):,}")
print(f"Train at-bats:   {len(train_abs):,}  ({train_mask.sum():,} pitches)")
print(f"Test  at-bats:   {len(test_abs):,}  ({test_mask.sum():,} pitches)")
print(f"\nData density check:")
print(f"  Memoryless: {train_mask.sum():,} pitches / {state_space:,} states = "
      f"{train_mask.sum() / state_space:,.0f} obs/state  ← excellent ✓")
print(f"  Augmented:  {train_mask.sum():,} pitches / {state_space * (N_ACTIONS + 1):,} states = "
      f"{train_mask.sum() / (state_space * (N_ACTIONS + 1)):,.1f} obs/state  ← solid ✓")
print(f"\nBoth well above the MIN_VISITS=5 fallback threshold.")
print(f"Expect very low fallback rates for both agents.")


### Model 1: Tabular Q-Learning (Memoryless State)

**Approach:** We implement the classic Q-learning update rule:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\bigr]$$

**State:** (balls, strikes, on_1b, on_2b, on_3b, batter_righty, pitcher_righty) — 7 components, fully discrete.

**Action:** Pitch type index (0–N).

**Reward:** $-\Delta\text{RunExp}$ per pitch — the pitcher wants to *lower* run expectancy.

**Why this model first?** It establishes a clean, interpretable baseline that ignores pitch history. This lets us quantify exactly how much the previous pitch matters when we add it in Model 2.

In [ ]:
# ── Q-Learning Agent ─────────────────────────────────────────────────────────
#
# Tabular Q-learning with:
#   1. Visit-count learning rate decay — α_t = alpha / sqrt(N(s,a))
#      Satisfies Robbins-Monro conditions → guaranteed convergence.
#      Rare pitch types stabilize slowly; common ones (FF) stabilize quickly.
#      Without this, the last few observations overwrite all prior learning.
#
#   2. Pessimistic initialization — Q = -0.01 for unseen (state, action) pairs
#      Agent prefers any observed pitch over an unobserved one.
#      Prevents the argmax from arbitrarily picking action 0 for new states.
#
#   3. Empirical fallback — tracks what MLB actually throws per state.
#      Used when total visits < min_visits (state seen too rarely to trust Q).
#      Fallback returns the MLB most-common pitch, not a random Q-value.
#
# BELLMAN UPDATE:
#   Q(s,a) ← Q(s,a) + α_t · [r + γ·max_a' Q(s',a') − Q(s,a)]
#   When done=True (last pitch of AB): target = r only, no next-state bootstrap.
#
# TRAINING IS UNMASKED — all 18 pitch types are valid actions.
# The agent learns from every pitch MLB threw; data naturally limits what
# appears in each pitcher's Q-table entries.

MIN_VISITS_FOR_INFERENCE = 5

class QLearningAgent:
    """
    Tabular Q-learning agent for pitch sequencing.

    Two inference modes:
      best_action(state)         — unmasked, all pitch types
      best_action_masked(state)  — restricted to allowed set (for dashboard use)
    """

    def __init__(self, n_actions, alpha=0.1, gamma=0.95, init_q=-0.01,
                 min_visits=MIN_VISITS_FOR_INFERENCE):
        self.n_actions  = n_actions
        self.alpha      = alpha
        self.gamma      = gamma
        self.init_q     = init_q
        self.min_visits = min_visits
        self.Q          = defaultdict(lambda: self.init_q)
        self.N          = defaultdict(int)
        self.empirical  = defaultdict(lambda: defaultdict(int))

    def _alpha_t(self, s, a):
        return self.alpha / np.sqrt(max(1, self.N[(s, a)]))

    def _total_visits(self, s):
        return sum(self.N.get((s, a), 0) for a in range(self.n_actions))

    def _empirical_fallback(self, s):
        """Return most-common MLB pitch for this state; default to FF."""
        counts = self.empirical[s]
        if not counts:
            return PITCH_TO_IDX.get('FF', 0)
        return max(counts, key=counts.get)

    def best_action(self, state_tuple):
        """
        Greedy inference. Uses empirical fallback if state has < min_visits.
        This prevents the -0.01 init value from being returned as a real recommendation.
        """
        if self._total_visits(state_tuple) < self.min_visits:
            return self._empirical_fallback(state_tuple)
        return int(max(range(self.n_actions),
                       key=lambda a: self.Q[(state_tuple, a)]))

    def best_q(self, state_tuple):
        return max(self.Q[(state_tuple, a)] for a in range(self.n_actions))

    def train_from_arrays(self, s_arr, a_arr, r_arr, ns_arr, done_arr,
                          n_epochs=8, shuffle=True):
        """
        Offline batch Q-learning from pre-built numpy arrays.
        Epoch 0 also builds the empirical policy (MLB actual pitch choices per state).
        """
        n = len(a_arr)
        idx = np.arange(n, dtype=np.int64)
        losses = []

        for epoch in range(n_epochs):
            if shuffle:
                np.random.shuffle(idx)
            epoch_loss = 0.0

            for i in idx:
                s    = tuple(s_arr[i])
                a    = int(a_arr[i])
                r    = float(r_arr[i])
                ns   = tuple(ns_arr[i])
                done = bool(done_arr[i])

                if epoch == 0:
                    self.empirical[s][a] += 1

                self.N[(s, a)] += 1
                alpha_t = self._alpha_t(s, a)
                old_q   = self.Q[(s, a)]
                target  = r if done else r + self.gamma * self.best_q(ns)
                self.Q[(s, a)] += alpha_t * (target - old_q)
                epoch_loss += (self.Q[(s, a)] - old_q) ** 2

            losses.append(epoch_loss / n)
            print(f"  Epoch {epoch+1}/{n_epochs}  MSE-delta: {losses[-1]:.6f}")

        return losses

print(f"QLearningAgent defined.")
print(f"  alpha=0.1 (decays as alpha/sqrt(N))")
print(f"  gamma=0.95, init_q=-0.01, min_visits={MIN_VISITS_FOR_INFERENCE}")


In [ ]:
# ── Train Memoryless Q-Agent ─────────────────────────────────────────────────
# State: (balls, strikes, outs, on_1b, on_2b, on_3b, batter_righty, pitcher_righty)
# 8 dimensions, 1,152 theoretical states.
# This agent has NO knowledge of what was thrown previously.
# It learns only from the current game situation.

agent_ml = QLearningAgent(n_actions=N_ACTIONS, alpha=0.1, gamma=0.95, init_q=-0.01)

print("Training Memoryless Q-Learning Agent...")
print(f"State: {STATE_COLS}")
print(f"State space: {state_space:,} theoretical states")
print(f"Training on {train_mask.sum():,} transitions across 8 epochs.\n")

losses_ml = agent_ml.train_from_arrays(
    s_arr    = state_arr[train_mask],
    a_arr    = action_arr[train_mask],
    r_arr    = reward_arr[train_mask],
    ns_arr   = next_state_arr[train_mask],
    done_arr = done_arr[train_mask],
    n_epochs = 8,
    shuffle  = True
)

print(f"\nQ-table entries: {len(agent_ml.Q):,}")
print(f"Empirical states populated: {len(agent_ml.empirical):,}")
print(f"Most visited (s,a) pairs: {sorted(agent_ml.N.values(), reverse=True)[:5]}")
print("Training complete. ✓")


In [ ]:
# ── Visualize Training Convergence ───────────────────────────────────────
# A declining MSE-delta means Q-values are stabilizing — the agent has converged.

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses_ml)+1), losses_ml, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Q-Update')
plt.title('Memoryless Q-Agent: Training Convergence')
plt.xticks(range(1, len(losses_ml)+1))
plt.tight_layout()
plt.show()

print("Interpretation: Declining MSE-delta indicates Q-values are stabilizing across epochs.")


In [ ]:
# ── Evaluate Agent vs Historical MLB Policy ───────────────────────────────────
# Evaluation is done UNFILTERED — the agent competes across all 18 pitch types.
# The Q-advantage metric is honest: it reflects how often the agent's choice
# (whatever it is) beats the MLB pitcher's actual choice in Q-value terms.
#
# DISPLAY_PITCHES: used only in policy tables below, not in evaluation.
# Restricts printed recommendations to pitches thrown >= 50,000 times.
# This removes knuckleball (KN=1,201), eephus (EP=1,371), screwball (SC=189),
# pitchouts (PO=112), and Statcast noise types (CS, FA, FO, UN).
# These appear in the Q-table with inflated values due to action selection bias:
# specialists throw rare pitches in cherry-picked situations → high Q in those spots.
# The filter doesn't change the model — it cleans up what we print.

# Display/evaluation pitch set matches the website action space. FS is intentionally kept.
DISPLAY_PITCHES = [p for p in WEBSITE_PITCHES if p in PITCH_TO_IDX]
print(f"Website pitch types for display/export: {DISPLAY_PITCHES}")
print(f"Excluded from modeling: rare/noisy pitch codes outside {WEBSITE_PITCHES}")
DISPLAY_PITCH_SET = set(PITCH_TO_IDX[p] for p in DISPLAY_PITCHES)

def evaluate_agent(agent, s_arr, a_mlb_arr, r_arr, ns_arr, done_arr, idx_to_pitch):
    """
    Evaluate agent vs historical MLB policy on held-out test set.
    Evaluation is UNFILTERED — all pitch types compete.
    The Q-advantage metric is the core result: positive = agent beats MLB.
    """
    n = len(a_mlb_arr)
    q_agent_vals  = np.empty(n)
    q_mlb_vals    = np.empty(n)
    agreements    = np.empty(n, dtype=int)
    q_advantages  = np.empty(n)
    fallback_hits = 0

    for i in range(n):
        s     = tuple(s_arr[i])
        a_mlb = int(a_mlb_arr[i])

        a_agent = agent.best_action(s)
        if agent._total_visits(s) < agent.min_visits:
            fallback_hits += 1

        q_agent = agent.Q[(s, a_agent)]
        q_mlb   = agent.Q[(s, a_mlb)]

        q_agent_vals[i] = q_agent
        q_mlb_vals[i]   = q_mlb
        agreements[i]   = int(a_agent == a_mlb)
        q_advantages[i] = q_agent - q_mlb

    return {
        'mean_q_agent'    : float(np.mean(q_agent_vals)),
        'mean_q_mlb'      : float(np.mean(q_mlb_vals)),
        'agreement_rate'  : float(np.mean(agreements)),
        'mean_q_advantage': float(np.mean(q_advantages)),
        'fallback_rate'   : fallback_hits / n,
        'n_test'          : n,
    }

metrics_ml = evaluate_agent(
    agent_ml,
    state_arr[test_mask], action_arr[test_mask],
    reward_arr[test_mask], next_state_arr[test_mask], done_arr[test_mask],
    IDX_TO_PITCH
)

print("\n=== Memoryless Q-Agent Evaluation (full unfiltered test set) ===")
for k, v in metrics_ml.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    elif isinstance(v, int):
        print(f"  {k:25s}: {v:,}")
    else:
        print(f"  {k:25s}: {v}")


### Model 2: History-Augmented Q-Learning (Tunneling-Aware State)

**Approach:** Identical Q-learning update rule, but the state now includes the **previous pitch type** as an additional component:

$$s' = (\text{balls}, \text{strikes}, \text{on\_1b}, \text{on\_2b}, \text{on\_3b}, \text{batter\_R}, \text{pitcher\_R}, \text{prev\_pitch\_idx})$$

**Why this matters — tunneling:** In pitching, a 98 mph 4-seam fastball followed by a slider with a similar release point and first-25-feet trajectory is far more effective than the same slider thrown cold. The batter's internal timing is calibrated to fastball speed, making the speed change more deceptive. By including `prev_pitch_idx` in the state, the agent can learn exactly these *conditional* effectiveness patterns — e.g., that SL after FF earns more reward than SL after SL.

**Expected improvement over Model 1:** We expect higher Q-value advantage (agent beats MLB more often) because the agent can learn pitch-pair synergies that the memoryless agent misses.

In [ ]:
# ── Train History-Augmented Q-Agent ──────────────────────────────────────────
# State: same 8D base + prev_pitch_action_idx appended as 9th dimension.
# prev_pitch = -1 sentinel for first pitch of AB (no prior pitch).
#
# This agent learns whether knowing the previous pitch TYPE changes what
# the optimal next pitch is — the core tunneling hypothesis.
#
# Example: after a 98 mph fastball (FF), a slider (SL) breaking away from
# the same release point is harder to hit than SL after SL — because the
# batter's internal timing is set to fastball speed. If the agent learns
# this, it will show different Q-values for (0-2, FF_prev → SL) vs
# (0-2, SL_prev → SL). That difference is tunneling.

agent_aug = QLearningAgent(n_actions=N_ACTIONS, alpha=0.1, gamma=0.95, init_q=-0.01)

print("Training History-Augmented Q-Learning Agent...")
print(f"Base state: {STATE_COLS}")
print(f"Augmented state: {aug_state_arr.shape[1]}D (base {state_arr.shape[1]} + prev_pitch)")
print(f"Training on {train_mask.sum():,} transitions across 8 epochs.\n")

losses_aug = agent_aug.train_from_arrays(
    s_arr    = aug_state_arr[train_mask],
    a_arr    = action_arr[train_mask],
    r_arr    = reward_arr[train_mask],
    ns_arr   = aug_next_state_arr[train_mask],
    done_arr = done_arr[train_mask],
    n_epochs = 8,
    shuffle  = True
)

print(f"\nQ-table entries: {len(agent_aug.Q):,}")
print(f"Empirical states populated: {len(agent_aug.empirical):,}")
print("Training complete. ✓")


In [ ]:
# ── Convergence Comparison + Augmented Agent Evaluation ──────────────────────

plt.figure(figsize=(9, 4))
plt.plot(range(1, 9), losses_ml,  marker='o', label='Memoryless (no history)',
         color='steelblue', linewidth=2)
plt.plot(range(1, 9), losses_aug, marker='s', label='History-Augmented (prev pitch)',
         color='coral', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Q-Update (lower = more stable)')
plt.title('Training Convergence: Memoryless vs History-Augmented Q-Agent')
plt.legend()
plt.xticks(range(1, 9))
plt.tight_layout()
plt.show()
print("Both declining curves = both agents converged. Augmented starts higher")
print("because it has more unique states to fill (8D base × 19 prev_pitch values).")

metrics_aug = evaluate_agent(
    agent_aug,
    aug_state_arr[test_mask], action_arr[test_mask],
    reward_arr[test_mask], aug_next_state_arr[test_mask], done_arr[test_mask],
    IDX_TO_PITCH
)

print("\n=== History-Augmented Q-Agent Evaluation ===")
for k, v in metrics_aug.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    elif isinstance(v, int):
        print(f"  {k:25s}: {v:,}")
    else:
        print(f"  {k:25s}: {v}")


### Model Comparison

In [ ]:
# ── Side-by-Side Model Comparison ────────────────────────────────────────────
# This is the core result of the project.
# Both agents are evaluated on the same 299k held-out test pitches.
# The difference in Q-advantage between them measures the value of pitch history.

results_df = pd.DataFrame([
    {
        'Model'           : 'Memoryless Q-Agent',
        'State Dims'      : state_arr.shape[1],
        'Mean Q (Agent)'  : metrics_ml['mean_q_agent'],
        'Mean Q (MLB)'    : metrics_ml['mean_q_mlb'],
        'Q Advantage'     : metrics_ml['mean_q_advantage'],
        'Agreement Rate'  : metrics_ml['agreement_rate'],
        'Fallback Rate'   : metrics_ml['fallback_rate'],
        'Q-Table Entries' : len(agent_ml.Q),
    },
    {
        'Model'           : 'History-Augmented Q-Agent',
        'State Dims'      : aug_state_arr.shape[1],
        'Mean Q (Agent)'  : metrics_aug['mean_q_agent'],
        'Mean Q (MLB)'    : metrics_aug['mean_q_mlb'],
        'Q Advantage'     : metrics_aug['mean_q_advantage'],
        'Agreement Rate'  : metrics_aug['agreement_rate'],
        'Fallback Rate'   : metrics_aug['fallback_rate'],
        'Q-Table Entries' : len(agent_aug.Q),
    },
])
display(results_df.set_index('Model'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(results_df['Model'], results_df['Q Advantage'],
            color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Q-Value Advantage over MLB')
axes[0].set_ylabel('Mean Q(agent) – Q(MLB)')
axes[0].axhline(0, color='black', linewidth=0.8)
for i, v in enumerate(results_df['Q Advantage']):
    axes[0].text(i, v + 0.0005, f"{v:.4f}", ha='center', fontsize=10, fontweight='bold')

axes[1].bar(results_df['Model'], results_df['Agreement Rate'],
            color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title('Policy Agreement with MLB')
axes[1].set_ylabel('Fraction where agent = MLB choice')
axes[1].set_ylim(0, 0.6)

axes[2].bar(results_df['Model'], results_df['Fallback Rate'],
            color=['steelblue', 'coral'], edgecolor='black')
axes[2].set_title('Fallback Rate')
axes[2].set_ylabel('Fraction using empirical fallback')
axes[2].set_ylim(0, 0.15)

plt.suptitle('Model Comparison: Memoryless vs History-Augmented Q-Agent', fontsize=13)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Q Advantage > 0: both agents find better pitches than historical MLB average.")
print("  Higher Q Advantage in augmented: knowing prev pitch improves pitch selection.")
print("  Agreement Rate ~14-20%: agent deliberately diverges from MLB (that's the goal).")
print("  Fallback Rate near 0: excellent data density — Q-values are reliable.")


### Policy Interpretation: What Did the Agent Learn?

In [ ]:
# ── Policy Interpretation: What Did the Agent Learn? ─────────────────────────
#
# The policy table is filtered to DISPLAY_PITCHES (>= 50k thrown) so
# recommendations only show genuine primary pitch types.
#
# best_action_filtered() picks the highest-Q pitch that is in DISPLAY_PITCHES.
# This mirrors how real pitchers work — a knuckleballer throws KN because
# it's their specialty, not because Q(state, KN) > Q(state, FF) globally.
# The filter removes action selection bias from the display without changing
# anything about the trained Q-table.

# MLB most-common pitch by (balls, strikes) for comparison
mlb_common = (
    mdp_df.groupby(['balls', 'strikes', 'pitch_type'])
    .size().reset_index(name='n')
)
mlb_common = (
    mlb_common.sort_values('n', ascending=False)
    .groupby(['balls', 'strikes']).first().reset_index()
    [['balls', 'strikes', 'pitch_type']]
)
mlb_dict = {(int(r.balls), int(r.strikes)): r.pitch_type
            for _, r in mlb_common.iterrows()}

def make_state(balls, strikes, outs, on1b, on2b, on3b, batter_r, pitcher_r):
    """Build memoryless state tuple in STATE_COLS order."""
    mapping = {
        'balls': balls, 'strikes': strikes, 'outs': outs,
        'on_1b': on1b, 'on_2b': on2b, 'on_3b': on3b,
        'batter_righty': batter_r, 'pitcher_righty': pitcher_r,
    }
    return tuple(mapping.get(c, 0) for c in STATE_COLS)

def make_aug_state(balls, strikes, outs, on1b, on2b, on3b, batter_r, pitcher_r,
                   prev_pitch_idx=-1):
    """Build augmented state tuple: base state + prev_pitch appended."""
    return make_state(balls, strikes, outs, on1b, on2b, on3b, batter_r, pitcher_r)            + (prev_pitch_idx,)

def best_action_filtered(agent, state_tuple, allowed=None):
    """
    Return the highest-Q pitch restricted to DISPLAY_PITCHES (primary pitch types).
    Falls back to empirical MLB most-common if no display pitch has been visited.
    This ONLY affects what we print — the trained Q-table is unchanged.
    """
    filter_set = DISPLAY_PITCH_SET if allowed is None else allowed
    total_v = agent._total_visits(state_tuple)

    if total_v < agent.min_visits:
        # Empirical fallback restricted to display pitches
        counts = {a: c for a, c in agent.empirical[state_tuple].items()
                  if a in filter_set}
        if counts:
            return max(counts, key=counts.get)
        return PITCH_TO_IDX.get('FF', 6)

    return int(max(filter_set,
                   key=lambda a: agent.Q[(state_tuple, a)]))

# Policy table: first pitch of AB, outs=0, RHB vs RHP
runner_scenarios = [
    (0,0,0, "Empty"),
    (1,0,0, "1B only"),
    (0,1,0, "2B (RISP)"),
    (1,1,1, "Loaded"),
]

print("=== Agent vs MLB: First Pitch of AB (no prev pitch, outs=0, RHB vs RHP) ===")
print("(Recommendations filtered to primary pitch types — >= 50k thrown in data)")
print(f"{'Count':<8} {'Runners':<12} {'Memoryless':<12} {'Augmented':<12} {'MLB':>8} {'FB?'}")
print("-" * 65)
for b in range(4):
    for k in range(3):
        for on1, on2, on3, rlabel in runner_scenarios:
            s_ml  = make_state(b, k, 0, on1, on2, on3, 1, 1)
            s_aug = make_aug_state(b, k, 0, on1, on2, on3, 1, 1, prev_pitch_idx=-1)

            rec_ml  = IDX_TO_PITCH.get(best_action_filtered(agent_ml, s_ml),  '?')
            rec_aug = IDX_TO_PITCH.get(best_action_filtered(agent_aug, s_aug), '?')
            mlb_p   = mlb_dict.get((b, k), '?')

            fb_ml  = agent_ml._total_visits(s_ml)  < MIN_VISITS_FOR_INFERENCE
            fb_aug = agent_aug._total_visits(s_aug) < MIN_VISITS_FOR_INFERENCE
            fb_str = ('FB-ml ' if fb_ml else '') + ('FB-aug' if fb_aug else '')

            print(f"{b}-{k:<6} {rlabel:<12} {rec_ml:<12} {rec_aug:<12} {mlb_p:>8} {fb_str}")
    print()

print("NOTE: Both agents were trained on ALL 18 pitch types — filtering is display only.")
print("KN/EP/SC were excluded because their Q-values are inflated by action selection")
print("bias (specialists use rare pitches in cherry-picked situations → artificially high Q).")


In [ ]:
# ── Tunneling Effect: How Does Previous Pitch Change Recommendations? ─────────
#
# Fix situation: 0-2 count, 0 outs, bases empty, RHB vs RHP.
# Vary only the previous pitch type.
#
# KEY: memoryless gives the SAME recommendation regardless of prev pitch.
# Augmented gives DIFFERENT recommendations — that IS the tunneling effect.
#
# Filtered to primary pitch types for clean display.

print("=== Tunneling Effect: 0-2 Count, Bases Empty, RHB vs RHP, 0 Outs ===")
print("(Filtered to primary pitch types >= 50k thrown)")
print(f"{'Prev Pitch':<12} {'Memoryless':<14} {'Q':>8}  {'Augmented':<14} {'Q':>8}  {'Same?'}")
print("-" * 72)

s_ml_02 = make_state(0, 2, 0, 0, 0, 0, 1, 1)   # memoryless: always same state

prev_pitches = [p for p in DISPLAY_PITCHES if p in PITCH_TO_IDX]
for prev_p in prev_pitches:
    prev_idx = PITCH_TO_IDX[prev_p]
    s_aug_02 = make_aug_state(0, 2, 0, 0, 0, 0, 1, 1, prev_pitch_idx=prev_idx)

    a_ml  = best_action_filtered(agent_ml, s_ml_02)
    a_aug = best_action_filtered(agent_aug, s_aug_02)
    q_ml  = agent_ml.Q[(s_ml_02, a_ml)]
    q_aug = agent_aug.Q[(s_aug_02, a_aug)]

    rec_ml  = IDX_TO_PITCH.get(a_ml, '?')
    rec_aug = IDX_TO_PITCH.get(a_aug, '?')
    same    = '← same' if rec_ml == rec_aug else '← DIFFERS'

    print(f"{prev_p:<12} {rec_ml:<14} {q_ml:>8.4f}  {rec_aug:<14} {q_aug:>8.4f}  {same}")

print()
print("← DIFFERS rows = augmented agent learned that prev pitch changes optimal choice.")
print("   Memoryless always gives the same answer — it has no concept of what came before.")
print("   The variation in the Augmented column IS the tunneling effect.")


In [ ]:
# ── Q-Value Heatmap: Tunneling in the Augmented Agent ────────────────────────
# State: 0-2 count, bases empty, RHB vs RHP, 0 outs.
# Rows = previous pitch thrown. Columns = pitch being evaluated.
# Value = Q(state_with_prev_pitch, next_pitch_action).
#
# High Q values off the main diagonal = agent prefers DIFFERENT pitch after X
# than it does after Y — that's the conditional sequencing signal.
#
# Compare to the memoryless heatmap (all rows identical) to see the difference.

top_pitches = [p for p in ['FF', 'SI', 'SL', 'CH', 'CU', 'FC', 'ST'] if p in PITCH_TO_IDX]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Augmented agent heatmap ──
q_matrix_aug = np.zeros((len(top_pitches), len(top_pitches)))
for i, prev_p in enumerate(top_pitches):
    prev_idx = PITCH_TO_IDX[prev_p]
    state = make_aug_state(0, 2, 0, 0, 0, 0, 1, 1, prev_pitch_idx=prev_idx)
    for j, next_p in enumerate(top_pitches):
        q_matrix_aug[i, j] = agent_aug.Q[(state, PITCH_TO_IDX[next_p])]

sns.heatmap(
    pd.DataFrame(q_matrix_aug, index=top_pitches, columns=top_pitches),
    annot=True, fmt='.3f', cmap='RdYlGn', linewidths=0.5, center=0,
    ax=axes[0], cbar_kws={'label': 'Q-Value'}
)
axes[0].set_title('History-Augmented Agent\n(Q varies by prev pitch → tunneling learned)')
axes[0].set_xlabel('Next Pitch (Action)')
axes[0].set_ylabel('Previous Pitch (State)')

# ── Memoryless agent heatmap (all rows identical — no prev pitch in state) ──
q_matrix_ml = np.zeros((len(top_pitches), len(top_pitches)))
s_ml = make_state(0, 2, 0, 0, 0, 0, 1, 1)
for i, prev_p in enumerate(top_pitches):   # prev_p ignored — state is same
    for j, next_p in enumerate(top_pitches):
        q_matrix_ml[i, j] = agent_ml.Q[(s_ml, PITCH_TO_IDX[next_p])]

sns.heatmap(
    pd.DataFrame(q_matrix_ml, index=top_pitches, columns=top_pitches),
    annot=True, fmt='.3f', cmap='RdYlGn', linewidths=0.5, center=0,
    ax=axes[1], cbar_kws={'label': 'Q-Value'}
)
axes[1].set_title('Memoryless Agent\n(All rows identical — no prev pitch in state)')
axes[1].set_xlabel('Next Pitch (Action)')
axes[1].set_ylabel('Previous Pitch (shown for comparison)')

plt.suptitle(
    'Q-Values: 0-2 Count, Bases Empty, RHB vs RHP, 0 Outs\n'
    'Row variation in left panel = tunneling effect learned by augmented agent',
    fontsize=12
)
plt.tight_layout()
plt.show()

print("LEFT: rows differ = agent learned prev pitch matters → tunneling effect captured")
print("RIGHT: all rows identical = memoryless agent has no sequencing awareness")
print("The difference between left and right is the core finding of this project.")


In [ ]:
# ── Count-Stratified Policy: Agent vs MLB ────────────────────────────────────
# Filtered to primary pitch types for clean, baseball-realistic display.

counts = [(b, k) for b in range(4) for k in range(3)]
rows_ml, rows_aug = [], []

for b, k in counts:
    s_ml  = make_state(b, k, 0, 0, 0, 0, 1, 1)
    s_aug = make_aug_state(b, k, 0, 0, 0, 0, 1, 1, prev_pitch_idx=-1)
    mlb_p = mlb_dict.get((b, k), None)

    a_ml  = best_action_filtered(agent_ml,  s_ml)
    a_aug = best_action_filtered(agent_aug, s_aug)
    q_ml  = agent_ml.best_q(s_ml)
    q_aug = agent_aug.best_q(s_aug)

    adv_ml = adv_aug = 0.0
    if mlb_p and mlb_p in PITCH_TO_IDX:
        adv_ml  = q_ml  - agent_ml.Q[(s_ml,  PITCH_TO_IDX[mlb_p])]
        adv_aug = q_aug - agent_aug.Q[(s_aug, PITCH_TO_IDX[mlb_p])]

    rows_ml.append({
        'Count': f"{b}-{k}",
        'Agent (filtered)': IDX_TO_PITCH.get(a_ml, '?'),
        'MLB': mlb_p or '?',
        'Q_Advantage': adv_ml
    })
    rows_aug.append({
        'Count': f"{b}-{k}",
        'Agent (filtered)': IDX_TO_PITCH.get(a_aug, '?'),
        'MLB': mlb_p or '?',
        'Q_Advantage': adv_aug
    })

policy_ml  = pd.DataFrame(rows_ml)
policy_aug = pd.DataFrame(rows_aug)

print("=== Memoryless Agent: Count Policy (first pitch, bases empty, RHB vs RHP) ===")
print("(Primary pitch types only — action selection bias removed from display)")
print(policy_ml.to_string(index=False))
print()
print("=== Augmented Agent: Count Policy (first pitch, bases empty, RHB vs RHP) ===")
print(policy_aug.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for policy, ax, title in [
    (policy_ml,  axes[0], 'Memoryless Agent'),
    (policy_aug, axes[1], 'History-Augmented Agent'),
]:
    colors = ['green' if x >= 0 else 'red' for x in policy['Q_Advantage']]
    bars = ax.bar(policy['Count'], policy['Q_Advantage'],
                  color=colors, edgecolor='black')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Count (Balls-Strikes)')
    ax.set_ylabel('Q-Advantage (Agent – MLB)')
    ax.set_title(f'{title}\nQ-Advantage over MLB by Count')
    ax.tick_params(axis='x', rotation=45)
    # Annotate each bar with the recommended pitch
    for bar, row in zip(bars, policy.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.0005 if bar.get_height() >= 0
                else bar.get_height() - 0.002,
                row._2,   # Agent (filtered) column
                ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('Q-Value Advantage over MLB Policy by Count (Primary Pitches Only)',
             fontsize=13)
plt.tight_layout()
plt.show()
print("\nBars annotated with the agent's filtered recommendation for each count.")
print("Positive (green) = agent finds a better pitch than MLB's historical choice.")


### Additional EDA Visualizations

In [ ]:
# ── 1. Pitch Distribution Overall ────────────────────────────────────────
pitch_counts = mdp_df['pitch_type'].value_counts()

plt.figure(figsize=(10, 4))
pitch_counts.plot(kind='bar', color=sns.color_palette('husl', len(pitch_counts)), edgecolor='black')
plt.title('Overall Pitch Type Distribution (2020–2024)')
plt.xlabel('Pitch Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ── 2. Delta Run Expectancy by Pitch Type ────────────────────────────────
# Negative values = pitcher-favorable (reduced run expectancy)

avg_dre = mdp_df.groupby('pitch_type')['delta_run_exp'].mean().sort_values()

plt.figure(figsize=(10, 4))
colors = ['green' if v < 0 else 'red' for v in avg_dre]
avg_dre.plot(kind='bar', color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Average Delta Run Expectancy by Pitch Type')
plt.xlabel('Pitch Type')
plt.ylabel('Avg Delta Run Exp (negative = good for pitcher)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ── 3. Pitch Sequencing Heatmap (MLB actual) ─────────────────────────────
# Transition matrix: given prev pitch (row), what does MLB throw next (col)?

if 'prev_pitch_type' in mdp_df.columns:
    seq_df = mdp_df[mdp_df['prev_pitch_type'].notna()]
    transition = pd.crosstab(
        seq_df['prev_pitch_type'],
        seq_df['pitch_type'],
        normalize='index'
    )

    plt.figure(figsize=(10, 7))
    sns.heatmap(transition, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.5,
                cbar_kws={'label': 'Transition Probability'})
    plt.title('MLB Pitch Sequencing: Transition Probabilities\n(Row = Previous Pitch, Col = Next Pitch)')
    plt.xlabel('Next Pitch')
    plt.ylabel('Previous Pitch')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 4. Top Pitch Sequences vs. Delta Run Expectancy ──────────────────────

if 'prev_pitch_type' in mdp_df.columns:
    seq_plot = mdp_df[
        mdp_df['prev_pitch_type'].notna() &
        mdp_df['pitch_type'].notna() &
        mdp_df['delta_run_exp'].notna()
    ].copy()

    seq_plot['sequence'] = seq_plot['prev_pitch_type'] + ' \u2192 ' + seq_plot['pitch_type']
    top15 = seq_plot['sequence'].value_counts().head(15).index
    seq_plot = seq_plot[seq_plot['sequence'].isin(top15)]

    avg_seq = (
        seq_plot.groupby('sequence')['delta_run_exp']
        .mean().sort_values().reset_index()
    )

    plt.figure(figsize=(12, 6))
    colors = ['green' if x < 0 else 'red' for x in avg_seq['delta_run_exp']]
    plt.barh(avg_seq['sequence'], avg_seq['delta_run_exp'], color=colors, edgecolor='black')
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title('Average Delta Run Expectancy by Two-Pitch Sequence (Top 15)')
    plt.xlabel('Avg Delta Run Expectancy (negative = pitcher-favorable)')
    plt.ylabel('Pitch Sequence')
    plt.tight_layout()
    plt.show()


**Modeling Results:**

**Model 1 — Memoryless Q-Agent:**
Trained on 1,196,025 pitches across 1,152 unique states (avg ~1,000 obs/state).
Zero fallback rate — every common game situation has reliable Q-values.
Q-advantage of +0.022 over the historical MLB policy on 299,221 held-out pitches.
Recommendations depend only on the current count, outs, base state, and handedness.
Because it cannot distinguish what came before, it gives the same recommendation
regardless of what the previous pitch was.

**Model 2 — History-Augmented Q-Agent:**
Same architecture, but state includes the previous pitch type as a 9th dimension.
19× larger state space (21,888 states), still well-covered at ~55 obs/state.
Q-advantage of +0.031 over MLB — a 38% improvement over the memoryless agent.
Fallback rate of 0.14% — essentially no states are data-starved.
The tunneling table shows the augmented agent giving different recommendations
for different previous pitches in the same count situation. The memoryless agent
gives the same answer every time — it has no concept of what came before.
This difference directly confirms the research hypothesis.

**Action Selection Bias (known limitation):**
Unfiltered policy tables showed rare specialist pitches (KN, EP) appearing as
recommendations in certain states. This is a known offline RL phenomenon —
knuckleball pitchers throw KN 100% of the time, and the data only records KN
being used in situations where specialists chose it deliberately. Q(state, KN)
inflates in those states because the data cannot distinguish "this pitch is
globally good" from "this pitch was selectively used by experts who knew when
to use it." Policy table displays are filtered to primary pitch types (≥50,000
thrown) to remove this. The Q-advantage evaluation remains unfiltered.


## Conclusions and Future Work

### Key Findings

1. **Both RL agents outperform the historical MLB average pitch selection** in
   expected run value on a held-out test set of 299,221 pitches, confirming
   that tabular Q-learning can learn meaningful pitch sequencing policies from
   Statcast data.

2. **Knowing the previous pitch type improves policy quality.** The history-augmented
   agent achieves a Q-advantage of +0.031 vs +0.022 for the memoryless agent —
   a 38% improvement attributable solely to the addition of pitch history.

3. **The tunneling effect is empirically confirmed.** The augmented agent's
   recommendations change based on what was thrown last pitch; the memoryless
   agent's do not. This difference in behavior is the model having learned that
   pitch sequences — not just individual pitch selection — drive run value.

4. **Data density matters more than state space richness.** The clean 8-dimensional
   base state (1,152 states, ~1,000 obs/state) produces a zero-fallback
   memoryless agent. Adding a single history dimension (previous pitch) expands
   the state space 19× while maintaining reliable Q-values throughout.

5. **Offline RL on historical data inherits action selection bias.** Rare pitch
   types thrown by specialists in cherry-picked situations show inflated Q-values
   in those states. Arsenal filtering at inference time is the correct remedy.

### Limitations

1. **Offline learning bias**: The agent can only learn about pitch sequences that
   MLB pitchers actually tried. It cannot discover novel sequences no pitcher
   has attempted. Its policy is bounded by the creativity of the historical data.

2. **Reward reflects outcomes, not pitch quality**. delta_run_exp measures what
   actually happened — a well-executed slider that gets hit by luck still
   generates a negative reward. The agent learns from results, not intent.

3. **Action selection bias**: Rare specialist pitches (knuckleball, eephus) appear
   in the Q-table with inflated values because they only appear in data when
   specialists deliberately chose them in favorable situations. Arsenal masking
   at inference time corrects this for practical use.

4. **Single year of data**: Training on 2024 only limits the agent's exposure to
   unusual but valid game situations. Multi-year data would reduce fallback rates
   further and reduce action selection bias for rare pitch types.

5. **No batter identity**: Two batters with the same handedness receive identical
   state representations regardless of their swing tendencies, chase rates, or
   hot/cold zones. The agent treats all left-handed batters as equivalent.

6. **No pitcher fatigue**: Pitch count, velocity decline, and stamina are not
   modeled. A pitcher throwing his 100th pitch and his 10th pitch face the same
   recommended sequence.

### Future Work

1. **Multi-year data (2020–2024)**: Five seasons would give ~7.5M pitches and
   ~6,200 obs/state in the augmented state space, effectively eliminating
   action selection bias for all pitch types including rare ones.

2. **Batter archetypes**: Cluster batters by O-Swing%, Z-Contact%, pull tendency,
   and hard-hit rate. Add a batter archetype index (0–3) as a state dimension.
   A high-chaser and a disciplined batter require completely different sequences
   even in identical count situations.

3. **Arsenal-masked inference**: At recommendation time, restrict the agent's action
   space to pitches the specific pitcher actually throws (≥5% of their outings).
   A pitcher with only FF and SL in their arsenal should never be recommended CU.
   The infrastructure for this was built and validated earlier in development.

4. **Deep Q-Network (DQN)**: As continuous kinematic features (release speed, spin
   rate, horizontal and vertical break) enter the state, the tabular approach
   cannot scale. A neural Q-function generalizes across similar-but-not-identical
   states and handles continuous inputs naturally.

5. **Pitcher fatigue**: Add pitch count in game as a state dimension. Pitchers lose
   velocity and movement as they tire — the agent should shift toward efficiency
   pitches late in counts and recommend more aggressive sequences early.


### Recommendations

- MLB pitching coaches could use the Q-value advantage maps to identify counts where their pitchers systematically deviate from optimal sequencing (e.g., over-relying on fastballs in 2-2 counts where the agent prefers off-speed).
- The tunneling matrix (Q-value heatmap of prev→next pitch) can serve as a pitch-calling cheat sheet tailored to specific count-batter combinations.


---

## References

1. LeDoux, J. (2017). *Introducing pybaseball: an Open Source Package for Baseball Data Analysis.* https://jamesrledoux.com/projects/open-source/introducing-pybaseball/

2. Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning, 8*(3–4), 279–292.

3. MLB Statcast. (2024). *Statcast Search.* https://baseballsavant.mlb.com/statcast_search

4. Franks, A., D'Amour, A., Cervone, D., & Bornn, L. (2016). Meta-analytics: tools for understanding the statistical properties of sports metrics. *Journal of Quantitative Analysis in Sports, 12*(4), 151–165.

5. Mnih, V., et al. (2015). Human-level control through deep reinforcement learning. *Nature, 518*, 529–533. (DQN reference for future work direction)

6. Sievert, C. (2019). *Expanding the Pitch Tunneling Concept.* Baseball Prospectus. https://www.baseballprospectus.com/
